# ML-07 — Baseline Action Score and Top-10 Review
Jackline Mutheu — Refresh / Content Opportunity Scoring

Working with an AI assistant: read `skills/README.md`, then load `building-baselines` + `flyrank/flyrank-data` — done before writing this notebook.


## 1. Two signal checks (before trusting a rule on them)

**Signal A — staleness vs. decline rate** (the signal behind FlyRank's refresh flags).
Claim: "Pages that haven't been updated in a while are more likely to be declining."

**Signal B — position vs. click-through rate** (the signal behind the CTR-fix logic).
Claim: "Pages ranking higher (lower position number) get a higher share of clicks per impression."

Both are checked as grouped bucket tables with `n` printed, using the weighted rate (total clicks / total impressions per bucket), not an average of per-row rates.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Label per data dictionary — trend_direction/trend_pct are the label source, NEVER features.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print("Rows:", len(df), "| overall decline rate:", round(df["is_declining_label"].mean(), 3))

# --- Signal A: freshness_tier vs decline rate ---
sig_a = df.groupby("freshness_tier").agg(
    n=("is_declining_label", "size"),
    decline_rate=("is_declining_label", "mean")
).round(3)
print("\n=== Signal A: freshness_tier vs decline rate ===")
print(sig_a)
print("Verdict: MIXED. Decline rate rises with staleness from 0-30 (0.511) through 31-90 (0.589)")
print("to 91-180 (0.611) as expected — but reverses at 181+ (0.471, n=174). That bucket clears")
print("the 50-row floor, so it's a real reversal, not noise wearing a costume. A clean staleness")
print("story doesn't hold at the far tail.")

# --- Signal B: position_tier vs weighted CTR ---
tier_stats = df.groupby("position_tier").agg(
    n=("content_id", "size"),
    total_clicks=("clicks_90d", "sum"),
    total_impr=("impressions_90d", "sum")
)
tier_stats["weighted_ctr_pct"] = (tier_stats["total_clicks"] / tier_stats["total_impr"] * 100).round(3)
print("\n=== Signal B: position_tier vs weighted CTR ===")
print(tier_stats[["n", "weighted_ctr_pct"]])
print("Verdict: CONFIRMED. CTR falls from top_3 (0.488%) down to deep (0.041%), an ~12x spread,")
print("with every bucket well above the 50-row floor. page_1 and striking sit close together")
print("(0.350% vs 0.347%) — a minor flat spot, but the overall shape holds.")


Rows: 30000 | overall decline rate: 0.542

=== Signal A: freshness_tier vs decline rate ===
                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
181+              174         0.471
31-90             175         0.589
91-180           9171         0.611
Verdict: MIXED. Decline rate rises with staleness from 0-30 (0.511) through 31-90 (0.589)
to 91-180 (0.611) as expected — but reverses at 181+ (0.471, n=174). That bucket clears
the 50-row floor, so it's a real reversal, not noise wearing a costume. A clean staleness
story doesn't hold at the far tail.

=== Signal B: position_tier vs weighted CTR ===
                   n  weighted_ctr_pct
position_tier                         
deep            1319             0.041
page_1         11814             0.350
page_3_5        7242             0.155
striking        7304             0.347
top_3           2321             0.488
Verdict: CONFIRMED. CTR falls from top_3 (0.488%) down to deep (0.

## 2. Build the ranked queue (writes the CSV)

**The rule, in plain words:** A page is worth a CTR-fix review if it gets enough impressions to matter, and its actual CTR sits well below what other pages in the *same position tier* typically earn — that gap is a fixable presentation problem (title/snippet), not a ranking problem. I leaned on Signal B (CONFIRMED) as the rule's foundation; Signal A (MIXED) told me staleness alone isn't a reliable single trigger, so it's not part of this rule's score — a clearly-explained "no" is still useful information, not a wasted check.

- **Visibility gate:** `impressions_90d >= 300` (matches the `moderate`+ threshold in `impression_tier`).
- **Expected CTR:** the weighted CTR for that page's own `position_tier` (Signal B's table).
- **Score:** `visible × max(expected_ctr − actual_ctr, 0) × impressions_90d` — readable on purpose, no fitted weights.
- **Reason code (one, for the whole rule):** `ctr_below_position_peer_expected`.
- **Action label:** `priority_review_ctr_fix` (top 50 by score), `monitor_ctr_fix` (rest of the flagged pages), `no_action` (no CTR gap).


In [2]:
tier_stats_map = tier_stats["weighted_ctr_pct"].to_dict()
df["expected_ctr_pct"] = df["position_tier"].map(tier_stats_map)
df["ctr_gap"] = df["expected_ctr_pct"] - df["ctr"]              # positive = underperforming peers
df["visible"] = (df["impressions_90d"] >= 300).astype(int)
df["ctr_gap_positive"] = df["ctr_gap"].clip(lower=0)
df["score"] = df["visible"] * df["ctr_gap_positive"] * df["impressions_90d"]

df["reason_code"] = np.where(
    (df["visible"] == 1) & (df["ctr_gap_positive"] > 0),
    "ctr_below_position_peer_expected",
    "no_action_trigger"
)

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)
N_PRIORITY = 50
ranked["action"] = "no_action"
ranked.loc[(ranked["score"] > 0) & (ranked.index < N_PRIORITY), "action"] = "priority_review_ctr_fix"
ranked.loc[(ranked["score"] > 0) & (ranked.index >= N_PRIORITY), "action"] = "monitor_ctr_fix"

print(ranked["action"].value_counts())
print(ranked["reason_code"].value_counts())

out_cols = ["content_id", "client_id", "position_tier", "impressions_90d", "ctr",
            "expected_ctr_pct", "ctr_gap", "score", "reason_code", "action"]
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("\nWrote", len(ranked), "rows to work/outputs/baseline_action_score.csv")


action
no_action                  16939
monitor_ctr_fix            13011
priority_review_ctr_fix       50
Name: count, dtype: int64
reason_code
no_action_trigger                   16939
ctr_below_position_peer_expected    13061
Name: count, dtype: int64



Wrote 30000 rows to work/outputs/baseline_action_score.csv


## 3. Top-10 review

For each of the top 10: the action, why it's there, and what would make it wrong.


In [3]:
top10 = ranked[out_cols].head(10)
print(top10.to_string())


             content_id          client_id position_tier  impressions_90d   ctr  expected_ctr_pct  ctr_gap       score                       reason_code                   action
0  content_8c19996aa890  client_4e07408562         top_3           509252  0.15             0.488    0.338  172127.176  ctr_below_position_peer_expected  priority_review_ctr_fix
1  content_8451fc6f034d  client_d029fa3a95         top_3           272144  0.03             0.488    0.458  124641.952  ctr_below_position_peer_expected  priority_review_ctr_fix
2  content_5fe46e04994d  client_4e07408562        page_1           517715  0.14             0.350    0.210  108720.150  ctr_below_position_peer_expected  priority_review_ctr_fix
3  content_36ff89c8214e  client_19581e27de        page_1           295097  0.05             0.350    0.300   88529.100  ctr_below_position_peer_expected  priority_review_ctr_fix
4  content_c8e9d6ab9013  client_19581e27de        page_1           208678  0.00             0.350    0.350   7

**Top-10, one line each:**

1. `content_8c19996aa890` (client_4e07408562, top_3) — **priority_review_ctr_fix**: 509k impressions but 0.15% CTR against a 0.49% peer expectation for top_3. *Wrong if:* this page's snippet is intentionally minimal (e.g. a definition box) that doesn't need a click to satisfy the searcher.
2. `content_8451fc6f034d` (client_d029fa3a95, top_3) — CTR 0.03% vs 0.49% expected. *Wrong if:* a rich result/answer box above it is absorbing the click.
3. `content_5fe46e04994d` (client_4e07408562, page_1) — same client as #1, 518k impressions, CTR 0.14% vs 0.35%. *Wrong if:* this is a duplicate/near-duplicate URL of #1 and the two shouldn't be reviewed as separate opportunities.
4. `content_36ff89c8214e` (client_19581e27de, page_1) — CTR 0.05% vs 0.35%. *Wrong if:* the keyword is branded/navigational, where low CTR is normal (searchers already know the destination).
5. `content_c8e9d6ab9013` (client_19581e27de, page_1) — **CTR is exactly 0.00%** on 208k impressions. *Wrong if:* click tracking is broken for this page/client rather than the title genuinely earning zero clicks — a 0% flat rate on this much volume is unusual enough to verify before treating it as a title problem.
6. `content_c84a0ab98e90` (client_f369cb89fc, page_1) — CTR 0.03% vs 0.35%. *Wrong if:* this page recently changed URL/title and hasn't accumulated a fair 90-day read yet.
7. `content_e12868d1f396` (client_4e07408562, top_3) — third appearance of this client in the top 10. *Wrong if:* one client's unusually large page sizes are dominating the list because the score multiplies by raw impressions — see the weak-pick note below.
8. `content_4a6607efcb46` (client_6208ef0f77, top_3) — CTR 0.01% vs 0.49%, the largest single gap in the list. *Wrong if:* this is a seasonal/trending page where the ranking spiked briefly and CTR hasn't caught up yet.
9. `content_cb112fce36be` (client_19581e27de, page_1) — CTR 0.16% vs 0.35%. *Wrong if:* this page's target intent is informational-only and a lower CTR reflects satisfied-without-clicking searchers, not a bad title.
10. `content_73c54f78c06a` (client_f369cb89fc, page_1) — CTR 0.10% vs 0.35%. *Wrong if:* the same client-level snippet/branding issue affects several of their pages at once, meaning the fix is a site-wide template problem, not a page-by-page one.


## 4. Weak picks + leakage check

**Weak pick, called out honestly:** three of the top 10 (#1, #3, #7) belong to the same client (`client_4e07408562`). That client has a median of ~2,677 impressions across its pages but a max of 517,715 — a few genuinely huge pages. Because the score multiplies by raw `impressions_90d`, one client with a wide internal spread can crowd out smaller clients with an equally real (proportionally larger) CTR gap. This is the score rewarding scale, not necessarily the size of the real opportunity — a fairer version would normalize the score within each client before ranking globally.

**Leakage check:**
- No `trend_direction` or `trend_pct` used anywhere in the score, gate, or reason code — those are the label source (per the data dictionary) and were excluded on purpose.
- No future/last-30d-vs-prev-30d comparison windows used — the rule only reads the current 90-day `ctr`, `impressions_90d`, and `position_tier`, all measured over the same trailing window, not a forward-looking one.
- `content_id` / `client_id` used only for grouping and the weak-pick check above, never as score inputs.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (client/content IDs are the repo's own pseudonyms)
- [x] My claims use careful words: observed, measured, weighted, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit repo URL on the card.
